# DeepSWE 5-Task: Qwen3.6-35B + Sharp Template (Colab A100)

**Runtime:** A100 40 GB, High-RAM. T4/L4 sind zu klein für 35B mit 16k Kontext.

**Einzige Stelle mit Key:** Zelle 2, ganz oben:

```python
MODAL_TOKEN_ID = ""       # <<< HIER Token ID einfügen
MODAL_TOKEN_SECRET = ""   # <<< HIER Token Secret einfügen
```

Danach einfach alle Zellen von oben nach unten ausführen. Das Notebook lädt das offizielle Qwen3.6-35B-UD-Q4_K_XL, baut llama.cpp, startet den Server mit HiveMinds Sharp-Template, öffnet einen cloudflared-Tunnel und lässt 5 DeepSWE-Tasks über Modal laufen.

Wenn du das Notebook teilst: Vorher Token löschen/rotieren.


> **Wichtig:** Dieses Notebook installiert **nicht** die HiveMind-Engine. Es startet llama.cpp mit dem offiziellen Qwen3.6-35B und HiveMinds Sharp-Template und benchmarkt mit DeepSWE/Pier + mini-swe-agent. Das Ergebnis ist eine Modell-/Template-Baseline, **kein HiveMind-Harness-Score**.


In [ ]:
# ===== ZELLE 2: HIER DEN MODAL-TOKEN EINTRAGEN =====
MODAL_TOKEN_ID = ""       # <<< HIER Token ID einfügen (beginnt meist mit ak-)
MODAL_TOKEN_SECRET = ""   # <<< HIER Token Secret einfügen (beginnt meist mit as-)

import os
if not MODAL_TOKEN_ID or not MODAL_TOKEN_SECRET:
    raise SystemExit("Trage MODAL_TOKEN_ID und MODAL_TOKEN_SECRET oben in Zelle 2 ein.")
os.environ["MODAL_TOKEN_ID"] = MODAL_TOKEN_ID
os.environ["MODAL_TOKEN_SECRET"] = MODAL_TOKEN_SECRET
print("Modal-Token gesetzt:", MODAL_TOKEN_ID[:8] + "...")


In [ ]:
!nvidia-smi
!df -h /
import os
os.environ["HF_HOME"] = "/content/hf"
!pip -q install -U huggingface_hub


In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download
repo = "unsloth/Qwen3.6-35B-A3B-GGUF"
files = list_repo_files(repo)
match = [f for f in files if f.lower().endswith("ud-q4_k_xl.gguf")]
assert match, f"no UD-Q4_K_XL gguf in {repo}"
model_path = hf_hub_download(repo_id=repo, filename=match[0], local_dir="/content/models")
print(model_path)


In [ ]:
!apt-get -qq update
!apt-get -qq install -y build-essential cmake git curl
!git clone --depth 1 --branch b12001 https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF
!cmake --build /content/llama.cpp/build --target llama-server -j$(nproc)


In [ ]:
!curl -fsSL https://raw.githubusercontent.com/BoredLuzo/HiveMind_llama/16fdd71/model_configs/chat_template_sharp_v22.5.0.jinja -o /content/sharp_v22.5.0.jinja
!wc -c /content/sharp_v22.5.0.jinja
import subprocess, time, urllib.request
log = open("/content/llama-server.log", "w")
proc = subprocess.Popen([
    "/content/llama.cpp/build/bin/llama-server",
    "-m", model_path, "--host", "0.0.0.0", "--port", "8080",
    "--ctx-size", "16384", "--n-gpu-layers", "99",
    "--flash-attn", "on", "--jinja",
    "--chat-template-file", "/content/sharp_v22.5.0.jinja",
    "--reasoning", "on", "--temp", "1.0", "--top-p", "0.95",
    "--top-k", "20", "--no-warmup",
], stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    try:
        urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2).read()
        break
    except Exception:
        time.sleep(2)
else:
    print(open("/content/llama-server.log").read()[-3000:])
    raise RuntimeError("llama-server nicht bereit (OOM? A100 40GB nehmen)")
print("llama-server ready on :8080")


In [ ]:
!pip -q install -U datacurve-pier modal
!git clone --depth 1 https://github.com/datacurve-ai/deep-swe.git /content/deep-swe
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared


In [ ]:
import subprocess, time, re, os
tlog = open("/content/cloudflared.log", "w")
tun = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8080"],
                       stdout=tlog, stderr=subprocess.STDOUT)
public = ""
for _ in range(60):
    time.sleep(1)
    text = open("/content/cloudflared.log").read()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if m:
        public = m.group(0)
        break
if not public:
    print(open("/content/cloudflared.log").read()[-2000:])
    raise RuntimeError("cloudflared hat keine URL erzeugt")
os.environ["DEEPSWE_MODEL_BASE_URL"] = public + "/v1"
print("public model endpoint:", os.environ["DEEPSWE_MODEL_BASE_URL"])


In [ ]:
!cd /content/deep-swe && pier run -p tasks --agent mini-swe-agent --model openai/qwen3.6-35b-a3b --env modal --n-tasks 5 --sample-seed 0 --n-concurrent 1 --jobs-dir /content/results/deepswe-qwen36 --yes --ae OPENAI_API_KEY=sk-local --ae OPENAI_BASE_URL=$DEEPSWE_MODEL_BASE_URL --ae OPENAI_API_BASE=$DEEPSWE_MODEL_BASE_URL --agent-timeout-multiplier 1.0


In [ ]:
# Aufraeumen
proc.terminate()
tun.terminate()
print("results:", "/content/results/deepswe-qwen36")
print(open("/content/llama-server.log").read()[-1500:])
